해당 지역의 좌표에 대응되는 H3 cell 찾기

In [1]:
from pathlib import Path
import pandas as pd
import h3

# =========================
# 1. 경로 설정
# =========================
BASE_DIR = Path.cwd()

# 파일 경로
candidate_path = BASE_DIR / "dataset" / "버티포트_일반_후보지.csv"
hub_path = BASE_DIR / "dataset" / "핵심허브_경도_위도_명칭.csv"
outgoing_path = BASE_DIR / "dataset" / "outgoing.csv"

# =========================
# 2. CSV 불러오기
# =========================
vertiport = pd.read_csv(candidate_path, encoding="cp949")
vertiport_hub = pd.read_csv(hub_path, encoding="cp949")

# outgoing.csv는 보통 UTF-8일 가능성이 높음
try:
    outgoing = pd.read_csv(outgoing_path, encoding="utf-8")
except UnicodeDecodeError:
    outgoing = pd.read_csv(outgoing_path, encoding="cp949")

# =========================
# 3. 후보지 + 허브 합치기
# =========================
distance_vertiport = pd.concat(
    [vertiport, vertiport_hub],
    ignore_index=True
)

# id를 0부터 다시 부여
distance_vertiport["id"] = range(len(distance_vertiport))

# =========================
# 4. H3 cell 생성
# =========================
resolution = 8

def latlon_to_h3(row):
    lat = row["y_latitude"]      # 위도
    lon = row["x_longtitude"]    # 경도

    # h3 버전에 따라 함수명이 다를 수 있음
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, resolution)
    else:
        return h3.geo_to_h3(lat, lon, resolution)

distance_vertiport["h3_cell"] = distance_vertiport.apply(
    latlon_to_h3,
    axis=1
)

# =========================
# 5. outgoing.csv에 존재하는 H3인지 확인
# =========================
valid_cells = set(outgoing["base_cell"]).union(set(outgoing["outgoing_cell"]))

distance_vertiport["exists_in_outgoing"] = distance_vertiport["h3_cell"].isin(valid_cells)

# =========================
# 6. 결과 출력
# =========================
result = distance_vertiport[
    ["id", "NAME", "y_latitude", "x_longtitude", "h3_cell", "exists_in_outgoing"]
]

print(result.to_string(index=False))

# =========================
# 7. CSV로 저장
# =========================
output_path = BASE_DIR / "dataset"/"h3_좌표_20개의_버티포트.csv"
result.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\n저장 완료: {output_path}")

 id             NAME  y_latitude  x_longtitude         h3_cell  exists_in_outgoing
  0   경기 성남시 분당구 서현동   37.359407    127.110785 8830e024dbfffff                True
  1       서울 강서구 화곡동   37.496731    126.868977 8830e0ac87fffff                True
  2        서울 양천구 목동   37.520998    126.836597 8830e0aea7fffff                True
  3       경기 부천시 오정동   37.528229    126.740430 8830e0a1b1fffff                True
  4   경기 수원시 영통구 원천동   37.258136    127.078535 8830e025a5fffff                True
  5   경기 안양시 동안구 평촌동   37.382887    126.967683 8830e035d7fffff                True
  6       서울 양천구 신정동   37.512807    126.878259 8830e0ac9dfffff                True
  7       서울 구로구 구로동   37.505937    126.871848 8830e0ac83fffff                True
  8       서울 강서구 등촌동   37.563371    126.841587 8830e0ae99fffff                True
  9        서울 양천구 목동   37.526261    126.879137 8830e1db25fffff                True
 10   경기 성남시 분당구 야탑동   37.370413    127.108489 8830e15361fffff                True
 11 

각 Cell에 대해 시간별 유출인구 분석

In [9]:
from pathlib import Path
import pandas as pd
import h3

BASE_DIR = Path.cwd()

export_cell = pd.read_csv(BASE_DIR/"dataset"/"h3_좌표_20개의_버티포트.csv")
outgoing_path = pd.read_csv(BASE_DIR/"dataset"/"outgoing.csv")

base_data = pd.DataFrame(outgoing_path)

# 1️⃣ 먼저 필터링
Designated_cell = base_data.loc[
    base_data["outgoing_cell"].isin(export_cell["h3_cell"])
]

# 2️⃣ h3_cell ↔ NAME 매핑 테이블 생성
mapping = export_cell[["h3_cell", "NAME"]]

# 3️⃣ merge 수행 (핵심)
Designated_cell = Designated_cell.merge(
    mapping,
    left_on="outgoing_cell",
    right_on="h3_cell",
    how="left"
)

# 필요 없는 컬럼 제거
Designated_cell = Designated_cell.drop(columns=["h3_cell"])

print(Designated_cell)

# 저장
output_path = BASE_DIR / "dataset"/"버티포트_지역의_시간대_별_인구유출_data.csv"
Designated_cell.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\n저장 완료: {output_path}")


             base_cell      time    outgoing_cell  outgoing_count  \
0      8830e004c9fffff  18:40:00  8830e0ad9bfffff              11   
1      8830e004d5fffff  19:00:00  8830e0af0bfffff              11   
2      8830e0061bfffff   7:40:00  8830e0af0bfffff              10   
3      8830e00697fffff  17:00:00  8830e0af6dfffff              10   
4      8830e006b7fffff  19:00:00  8830e03531fffff              10   
...                ...       ...              ...             ...   
13495  8830e17b65fffff  17:40:00  8830e15361fffff              14   
13496  8830e17b65fffff  18:40:00  8830e025a5fffff              21   
13497  8830e17b65fffff  18:40:00  8830e15361fffff              12   
13498  8830e17b65fffff  19:40:00  8830e025a5fffff              11   
13499  8830e17b65fffff  23:00:00  8830e025a5fffff              13   

       outgoing_time  outgoing_speed            NAME  
0              57.91           43.43    Gwang_Myeong  
1              45.73           55.38       경기 부천시 중동  
2     